## Task 4: Bubble Chart Analysis of App Size vs Rating
**Author:** Vansh Sharma

- Cleaned and merged the Play Store and User Reviews datasets.
- Filtered apps based on rating, installs, reviews, sentiment subjectivity, category, and app name.
- Translated selected categories for better visualization.
- Created a bubble chart showing the relationship between app size and average rating.
- Represented the number of installs using bubble size.
- Highlighted the **Game** category using a pink color.
- Displayed the graph only between **5 PM IST and 7 PM IST**.


# Import Libraries

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime
import pytz

# Load DataSet 1 `play_store_data`

In [12]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)

play_store_data.head()
play_store_data.columns
 


Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='object')

# Load DataSet 2 `review`


In [13]:
review = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\User Reviews (1).csv"
)

review.head()
review.columns

Index(['App', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity',
       'Sentiment_Subjectivity'],
      dtype='object')

### Data Cleaning

- Removed corrupted records from the dataset.
- Converted **Installs** and **Reviews** to numeric format.
- Converted **Last Updated** to datetime format.
- Filled missing values in **Rating**, **Current Ver**, **Android Ver**, and **Type**.
- Converted **Size** from KB/MB to MB.
- Replaced missing **Size** values with the median app size.

In [ ]:
# Remove corrupted row
play_store_data = play_store_data[
    play_store_data["Installs"] != "Free"
]

# Clean Installs column
play_store_data["Installs"] = (
    play_store_data["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(int)
)

# Convert Reviews into integer
play_store_data["Reviews"] = (
    play_store_data["Reviews"]
    .astype(int)
)

# Convert Last Updated into datetime
play_store_data["Last Updated"] = pd.to_datetime(
    play_store_data["Last Updated"],
    errors="coerce"
)

# Fill missing values
play_store_data["Rating"] = (
    play_store_data["Rating"].ffill()
)

play_store_data["Current Ver"] = (
    play_store_data["Current Ver"].ffill()
)

play_store_data["Android Ver"] = (
    play_store_data["Android Ver"].ffill()
)

play_store_data["Type"] = (
    play_store_data["Type"].fillna(
        play_store_data["Type"].mode()[0]
    )
)

# Convert Size into MB
play_store_data["Size"] = play_store_data["Size"].replace(
    "Varies with device", np.nan
)

# Convert KB to MB
play_store_data.loc[
    play_store_data["Size"].str.contains("k", na=False),
    "Size"
] = (
    play_store_data.loc[
        play_store_data["Size"].str.contains("k", na=False),
        "Size"
    ]
    .str.replace("k", "", regex=False)
    .astype(float)
    / 1024
)

# Convert MB values
play_store_data.loc[
    play_store_data["Size"].str.contains("M", na=False),
    "Size"
] = (
    play_store_data.loc[
        play_store_data["Size"].str.contains("M", na=False),
        "Size"
    ]
    .str.replace("M", "", regex=False)
    .astype(float)
)

# Convert Size to numeric
play_store_data["Size"] = pd.to_numeric(
    play_store_data["Size"],
    errors="coerce"
)

# Fill missing Size values with median
play_store_data["Size"] = (
    play_store_data["Size"]
    .fillna(play_store_data["Size"].median())
)



# Remove Duplicates From `play_store_data`

In [29]:
# Remove duplicate apps
play_store_data = play_store_data.drop_duplicates(
    subset="App",
    keep="first"
)

# Check
play_store_data.shape

(816, 14)

# Remove Unwanted Column From `review`

In [34]:
# Keep only required columns
review = review[
    ["App", "Sentiment_Subjectivity"]
]

# Remove missing values
review = review.dropna(
    subset=["Sentiment_Subjectivity"]
)

# Convert to float
review["Sentiment_Subjectivity"] = (
    review["Sentiment_Subjectivity"]
    .astype(float)
)

# If same app has multiple reviews, take average subjectivity
review = (
    review
    .groupby("App", as_index=False)
    .agg({
        "Sentiment_Subjectivity": "mean"
    })
)

# Check
review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 865 entries, 0 to 864
Data columns (total 2 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     865 non-null    object 
 1   Sentiment_Subjectivity  865 non-null    float64
dtypes: float64(1), object(1)
memory usage: 13.6+ KB


# Remove Duplicates From `review`

In [33]:
# Remove duplicate apps
review = review.drop_duplicates(
    subset="App",
    keep="first"
)

# Check
review.shape

(865, 2)

# Merge `play_store_data` and `review` DataSet

In [51]:
# Merge Play Store and Review datasets

play_store_data_merge = pd.merge(
    play_store_data,
    review,
    on="App",
    how="inner"
)

# Check
# play_store_data_merge.info()
play_store_data_merge.head(2)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Sentiment_Subjectivity_x,Sentiment_Subjectivity_y
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,0.641540,0.641540
1,Garden Coloring Book,ART_AND_DESIGN,4.4,13791,33.0,1000000,Free,0,Everyone,Art & Design,2017-09-20,2.9.2,3.0 and up,0.523447,0.523447


# Final DataSet After Merge = `play_store_data_merge'

In [42]:
play_store_data_merge.duplicated().sum()


np.int64(0)

### Data Filtering

- Selected apps with a **Rating greater than 3.5**.
- Included apps with **more than 500 reviews**.
- Filtered apps having **more than 50,000 installs**.
- Kept apps with **Sentiment Subjectivity greater than 0.5**.
- Excluded app names containing the letter **"S"**.
- Limited the analysis to the following categories:
  - Game
  - Beauty
  - Business
  - Comics
  - Communication
  - Dating
  - Entertainment
  - Social
  - Events

In [43]:
filtered = play_store_data[
    (play_store_data["Rating"] > 3.5) &
    (play_store_data["Reviews"] > 500) &
    (play_store_data["Installs"] > 50000) &
    (play_store_data["Sentiment_Subjectivity"] > 0.5) &
    (~play_store_data["App"].str.contains("S", case=False, na=False)) &
    (
        play_store_data["Category"].isin([
            "GAME",
            "BEAUTY",
            "BUSINESS",
            "COMICS",
            "COMMUNICATION",
            "DATING",
            "ENTERTAINMENT",
            "SOCIAL",
            "EVENTS"
        ])
    )
]

filtered.shape

(29, 14)

### Category Translation

- Translated **Beauty** to **Hindi (सौंदर्य)**.
- Translated **Business** to **Tamil (வணிகம்)**.
- Translated **Dating** to **German (Partnersuche)**.
- Updated category names for visualization while preserving the original analysis.

In [44]:
category_translation = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}

filtered["Category"] = (
    filtered["Category"]
    .replace(category_translation)
)

filtered.head()

C:\Users\Vansh Sharma\AppData\Local\Temp\ipykernel_20788\3807359468.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered["Category"] = (


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Sentiment_Subjectivity
42,Google Primer,வணிகம்,4.4,62272,18.0,10000000,Free,0,Everyone,Business,2018-06-26,3.550.2,4.1 and up,0.675000
45,Call Blocker,வணிகம்,4.6,188841,3.2,5000000,Free,0,Everyone,Business,2018-06-21,1.1.13,4.0 and up,0.655431
83,Chrome Dev,COMMUNICATION,4.4,63543,13.0,5000000,Free,0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,0.520779
87,GMX Mail,COMMUNICATION,4.3,258556,13.0,10000000,Free,0,Everyone,Communication,2018-07-25,Varies with device,Varies with device,0.540740
93,GroupMe,COMMUNICATION,4.5,330761,13.0,10000000,Free,0,Everyone,Communication,2018-07-03,Varies with device,Varies with device,0.518137


### Bubble Size Preparation

- Created a **Bubble_Size** column by scaling the number of installs.
- Used the scaled values to represent app popularity through bubble size in the chart.

In [45]:
filtered["Bubble_Size"] = (
    filtered["Installs"] / 10000
)

filtered.head()

C:\Users\Vansh Sharma\AppData\Local\Temp\ipykernel_20788\1257339984.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered["Bubble_Size"] = (


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Sentiment_Subjectivity,Bubble_Size
42,Google Primer,வணிகம்,4.4,62272,18.0,10000000,Free,0,Everyone,Business,2018-06-26,3.550.2,4.1 and up,0.675000,1000.0
45,Call Blocker,வணிகம்,4.6,188841,3.2,5000000,Free,0,Everyone,Business,2018-06-21,1.1.13,4.0 and up,0.655431,500.0
83,Chrome Dev,COMMUNICATION,4.4,63543,13.0,5000000,Free,0,Everyone,Communication,2018-08-02,69.0.3497.24,Varies with device,0.520779,500.0
87,GMX Mail,COMMUNICATION,4.3,258556,13.0,10000000,Free,0,Everyone,Communication,2018-07-25,Varies with device,Varies with device,0.540740,1000.0
93,GroupMe,COMMUNICATION,4.5,330761,13.0,10000000,Free,0,Everyone,Communication,2018-07-03,Varies with device,Varies with device,0.518137,1000.0


### Bubble Chart Visualization

- Created a bubble chart to analyze the relationship between **App Size (MB)** and **Average Rating**.
- Represented the **number of installs** using bubble size.
- Highlighted the **Game** category with a **pink** color.
- Displayed translated category names for **Beauty**, **Business**, and **Dating**.
- Added axis labels, legend, grid, and chart title for better readability.
- Configured the visualization to be available **only between 5:00 PM IST and 7:00 PM IST**.

In [48]:
# Current IST Time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# Allowed Time (5 PM - 7 PM)
start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    # Translate Categories
    category_translation = {
        "BEAUTY": "सौंदर्य",
        "BUSINESS": "வணிகம்",
        "DATING": "Partnersuche"
    }

    filtered["Category"] = (
        filtered["Category"]
        .replace(category_translation)
    )

    # Bubble Size
    filtered["Bubble_Size"] = (
        filtered["Installs"] / 10000
    )

    plt.figure(figsize=(14,7))

    for category in filtered["Category"].unique():

        data = filtered[
            filtered["Category"] == category
        ]

        # Highlight Game category in Pink
        if category == "GAME":
            color = "pink"
        else:
            color = None

        plt.scatter(
            data["Size"],
            data["Rating"],
            s=data["Bubble_Size"],
            c=color,
            alpha=0.6,
            edgecolors="black",
            label=category
        )

    plt.title(
        "Relationship Between App Size and Average Rating"
    )

    plt.xlabel("App Size (MB)")
    plt.ylabel("Average Rating")

    plt.legend(title="Category")

    plt.grid(True)

    plt.tight_layout()

    plt.show()

else:

    print(
        "Graph is available only between 5 PM IST and 7 PM IST"
    )

Graph is available only between 5 PM IST and 7 PM IST


# `KPIs`


In [49]:
# KPIs

total_apps = filtered["App"].nunique()

total_categories = filtered["Category"].nunique()

avg_rating = round(
    filtered["Rating"].mean(),
    2
)

avg_size = round(
    filtered["Size"].mean(),
    2
)

total_installs = filtered["Installs"].sum()

top_category = (
    filtered.groupby("Category")["Installs"]
    .sum()
    .idxmax()
)

print("Total Apps:", total_apps)
print("Total Categories:", total_categories)
print("Average Rating:", avg_rating)
print("Average App Size (MB):", avg_size)
print("Total Installs:", total_installs)
print("Top Category by Installs:", top_category)

Total Apps: 29
Total Categories: 6
Average Rating: 4.3
Average App Size (MB): 24.45
Total Installs: 471700000
Top Category by Installs: GAME


### Key Performance Indicators (KPIs)

- **Total Apps Analyzed:** 29

- **Total Categories:** 6

- **Average Rating:** 4.30

- **Average App Size:** 24.45 MB

- **Total Installs:** 471.70 Million

- **Top Category by Installs:** Game